# 09 — Closed-Loop Evaluation

**PLTMH–ELC–QLSTM**

Notebook ini menangani integrasi tiga strategi pengendalian:

- Fixed PI;
- LSTM–PI;
- QLSTM–PI;

ke simulator PLTMH–ELC yang sama.

Tahap ini menggunakan **model yang sudah dibekukan** dan tidak
membuka kembali proses pemilihan model.

Eksperimen final terdiri atas **12 deterministic closed-loop runs**:
empat skenario dan tiga controller.


## 1. Freeze Sebelum Hasil Closed-Loop

Sebelum hasil dilihat, workflow telah membekukan:

- scenario matrix;
- controller matrix;
- RK4 `dt = 0.0025 s`;
- observation cadence 20 Hz;
- gain-update cadence 10 Hz;
- gain envelope;
- gain-rate limiter;
- fallback gain;
- first adaptive update `2.6 s`;
- metric definitions;
- primary held-out families.

Adaptive gain tidak diubah sebelum atau pada saat gangguan.
Gain pertama dari scheduler baru dapat diterapkan pada `t = 2.6 s`.

Tidak dilakukan *post-result tuning*.


## 2. Catatan Rekonstruksi

Source asli Cell 151–155 telah disimpan pada:

`data/recovery/cell166_closed_loop_runtime_source_snapshot.json`

File tersebut mempertahankan kode scheduler, scenario freeze,
adaptive RK4 integration, dan eksperimen 12-run sebagaimana
dieksekusi pada runtime penelitian.

Mode default notebook migrasi adalah **verifikasi frozen evidence**.

Eksperimen tidak dijalankan ulang otomatis.


In [ ]:
# ============================================================
# 09.1 — PROJECT + CLOSED-LOOP ARTIFACT INTEGRITY
# ============================================================

from pathlib import Path

import hashlib
import json
import sys

import numpy as np
import pandas as pd


EXPECTED_UPSTREAM_CHECKPOINT = "563d98640c4b33966959bddabd507c702deec9fd"

EXPECTED_NOTEBOOK_08_SHA256 = (
    "2ca47ac12e4500683fac989c1fe0d857a0353d6b2de6117573ac15b04b1d0492"
)

EXPECTED_RUNTIME_SOURCE_SNAPSHOT_SHA256 = (
    "1c4e85e00ef351ee5c52fb930efe805ba35512a5ae3c9c80138ac53aa0e2947f"
)

EXPECTED_CLOSED_LOOP_ARTIFACT_SHA256 = (
    {
    "data/closed_loop/cell151_frozen_scheduler_policy.json": "c5071537ab73105f5481c871b921e5f0386699c43c898a54d8837069cea80fd5",
    "data/closed_loop/cell151_scheduler_smoke_audit.csv": "838d86902b52f917825bd3deeb7352dd9c140fc5639bd9edaa2b619a0e6956ef",
    "data/closed_loop/cell152_metric_definitions.csv": "e29f4025b5caec2e68779dededb7baa2da3e1015867c45cd5b5f254ae409013c",
    "data/closed_loop/cell152_pre_simulation_policy.json": "9ab3b1e070e2c7e7ad056f68e85aa0620364a5eb5ed12c285ddd883ec18281d4",
    "data/closed_loop/cell152_predefined_run_matrix.csv": "b3386a78be61793efe7591d7959472f39167159b50a35d4c0352b986735375b0",
    "data/closed_loop/cell152_predefined_scenario_matrix.csv": "5f0a31549e81fae9474cfc81221c455c5cd75e73d1aeadd605cb7439980ec527",
    "data/closed_loop/cell153_fixed_pi_structural_smoke.csv": "f966f60aa9a694813d53511a035dd6e988b0a2f4e7199a16d0d6b76e464e7e88",
    "data/closed_loop/cell153_physical_scenario_resolution.csv": "fbce994a8c4d3d946af0ea6ac08e111f6ea9920fbaf15f6e99049b8f20dc8f3a",
    "data/closed_loop/cell153_pi_output_bias_contract.csv": "4ac3a83b1cac7ab3cc546afd724866c52f2bd6328cdee07ad3ef06e07003ff1b",
    "data/closed_loop/cell153_simulator_readiness.json": "1f75fe58c40d8f371d194ed9bd71cf91cff835ffd332272106ac586a456d5e03",
    "data/closed_loop/cell154_adaptive_simulator_contract.json": "2a21ccb8a0a03f98c9498e6b9507633a4314e53b893a7bd49744fcb63e8695a1",
    "data/closed_loop/cell154_constant_gain_equivalence.csv": "008aef289cc95132799b093eaea75354c6bb568878e0f87c96bda9ec42f5f3d1",
    "data/closed_loop/cell154_scripted_gain_update_smoke.csv": "da5fc146dfd974dd967db33f836daeae2c453c639b27019cc5ffb88fb245b72d",
    "data/closed_loop/cell155_all_adaptive_gain_updates.csv.gz": "058ac58f1168dd85ae687ee40ace3e30de55863155716d46121fd1a84c042b84",
    "data/closed_loop/cell155_all_closed_loop_trajectories.csv.gz": "0a5a53a21f486c9020bcfbf7c236f739725e738af0181190f7e47c77336dba6b",
    "data/closed_loop/cell155_closed_loop_metrics.csv": "a9386b074dd5f5476b08439de2708826053d2c8db475233ea91f00bccd60c526",
    "data/closed_loop/cell155_comparative_experiment_metadata.json": "d5f155ba799aae3ac87f429d6a74947bda267f00d6dc638363af1d348ed31748",
    "data/closed_loop/cell155_primary_family_summary.csv": "e167773ea800de79b599196d79b94426121801ae9e68761bd2b3b16334dca1d5",
    "data/closed_loop/cell155_primary_family_winners.csv": "ad445a99769728349d3911892d423469d7556df5e2f701b8e9ef5a375d31e837",
    "data/closed_loop/cell155_scheduler_diagnostics.csv": "bb9198612a5750fbd64d2dd9364907b960e407e4ad664336660a7b20f5b0aa99"
}
)

CRITICAL_RELPATHS = (
    {
    "adaptive_contract": "data/closed_loop/cell154_adaptive_simulator_contract.json",
    "adaptive_updates": "data/closed_loop/cell155_all_adaptive_gain_updates.csv.gz",
    "constant_equivalence": "data/closed_loop/cell154_constant_gain_equivalence.csv",
    "experiment_metadata": "data/closed_loop/cell155_comparative_experiment_metadata.json",
    "metric_definitions": "data/closed_loop/cell152_metric_definitions.csv",
    "metrics": "data/closed_loop/cell155_closed_loop_metrics.csv",
    "physical_resolution": "data/closed_loop/cell153_physical_scenario_resolution.csv",
    "pre_simulation_policy": "data/closed_loop/cell152_pre_simulation_policy.json",
    "primary_summary": "data/closed_loop/cell155_primary_family_summary.csv",
    "primary_winners": "data/closed_loop/cell155_primary_family_winners.csv",
    "run_matrix": "data/closed_loop/cell152_predefined_run_matrix.csv",
    "scenario_matrix": "data/closed_loop/cell152_predefined_scenario_matrix.csv",
    "scheduler_diagnostics": "data/closed_loop/cell155_scheduler_diagnostics.csv",
    "scheduler_policy": "data/closed_loop/cell151_frozen_scheduler_policy.json",
    "scheduler_smoke": "data/closed_loop/cell151_scheduler_smoke_audit.csv",
    "scripted_gain_smoke": "data/closed_loop/cell154_scripted_gain_update_smoke.csv",
    "simulator_readiness": "data/closed_loop/cell153_simulator_readiness.json",
    "trajectories": "data/closed_loop/cell155_all_closed_loop_trajectories.csv.gz"
}
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            digest.update(chunk)

    return digest.hexdigest()


candidate_roots = [
    Path("/content/PLTMH-ELC-QLSTM"),
    Path.cwd(),
    Path.cwd().parent,
]


PROJECT_ROOT = None


for candidate in candidate_roots:

    candidate = candidate.resolve()

    if (
        (candidate / ".git").exists()
        and
        (candidate / "data").exists()
        and
        (candidate / "notebooks").exists()
    ):

        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:

    raise RuntimeError(
        "PLTMH-ELC-QLSTM repository root not found."
    )


if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


NOTEBOOK_08_PATH = (
    PROJECT_ROOT
    /
    "notebooks"
    /
    "08_train_qlstm.ipynb"
)


if (
    sha256_file(
        NOTEBOOK_08_PATH
    )
    !=
    EXPECTED_NOTEBOOK_08_SHA256
):

    raise RuntimeError(
        "Upstream Notebook 08 changed."
    )


RUNTIME_SOURCE_SNAPSHOT_PATH = (
    PROJECT_ROOT
    /
    "data"
    /
    "recovery"
    /
    "cell166_closed_loop_runtime_source_snapshot.json"
)


if (
    sha256_file(
        RUNTIME_SOURCE_SNAPSHOT_PATH
    )
    !=
    EXPECTED_RUNTIME_SOURCE_SNAPSHOT_SHA256
):

    raise RuntimeError(
        "Closed-loop runtime source snapshot changed."
    )


artifact_integrity = {}


for relative_path, expected_sha in (
    EXPECTED_CLOSED_LOOP_ARTIFACT_SHA256.items()
):

    path = (
        PROJECT_ROOT
        /
        relative_path
    )


    artifact_integrity[
        relative_path
    ] = bool(
        path.exists()
        and
        sha256_file(
            path
        )
        ==
        expected_sha
    )


for relative_path, valid in (
    artifact_integrity.items()
):

    print(
        f"{relative_path:76s}: {valid}"
    )


FROZEN_CLOSED_LOOP_ARTIFACTS_VALID = all(
    artifact_integrity.values()
)


if not FROZEN_CLOSED_LOOP_ARTIFACTS_VALID:

    raise RuntimeError(
        "Frozen closed-loop artifact changed."
    )


print(
    "\nFROZEN_CLOSED_LOOP_ARTIFACTS_VALID:",
    FROZEN_CLOSED_LOOP_ARTIFACTS_VALID
)


In [ ]:
# ============================================================
# 09.2 — FROZEN SCHEDULER POLICY
# ============================================================

EXPECTED_SCHEDULER_POLICY = {
    "admissible_gain_envelope": {
        "interpretation": "Finalized supervised-label envelope; not a universal physical stability guarantee.",
        "ki_max": 14.144,
        "ki_min": 3.0368,
        "kp_max": 3.36,
        "kp_min": 2.16
    },
    "cell151_data_access": {
        "test": false,
        "train": true,
        "validation": false
    },
    "fairness": {
        "same_bounds_lstm_qlstm": true,
        "same_fallback_lstm_qlstm": true,
        "same_rate_limit_lstm_qlstm": true,
        "same_scaler_lstm_qlstm": true,
        "same_update_cadence_lstm_qlstm": true
    },
    "fallback": {
        "ki": 5.72,
        "kp": 2.16,
        "transition": "same_gain_rate_limiter",
        "trigger": "nonfinite_or_failed_model_inference"
    },
    "fixed_pi": {
        "ki": 5.72,
        "kp": 2.16
    },
    "gain_update": {
        "between_updates": "ZERO_ORDER_HOLD",
        "observations_per_update": 2,
        "selected_using_closed_loop_results": false,
        "update_dt_s": 0.1,
        "update_rate_hz": 10.0
    },
    "input": {
        "feature_standardization": "CELL_138_TRAIN_ONLY_SCALER",
        "features": [
            "frequency_deviation_hz",
            "mechanical_power_kw",
            "electrical_power_kw",
            "dump_power_kw"
        ],
        "observation_dt_s": 0.05,
        "observation_rate_hz": 20.0,
        "window_points": 11,
        "window_span_s": 0.5
    },
    "model_artifacts": {
        "lstm": {
            "parameters": 5426,
            "path": "models/lstm_baseline/cell139_classical_lstm.pt",
            "sha256": "33fe34d8c06f2fdae316a302e75f5f398a9049e572093358dc77c3d8ee2d9a27"
        },
        "qlstm": {
            "parameters": 5422,
            "path": "models/qlstm/cell148_qlstm_best.pt",
            "quantum_parameters": 48,
            "sha256": "bf1a049a8bae552f5be7a3460507fe9d3912af7540ba617ea79623676862cfd7"
        }
    },
    "model_stage_checkpoint": "3787800b408e06f0855dda756c0cb4e0247c4982",
    "model_training_closed": true,
    "next_cell": "CELL_152_PREDEFINED_CLOSED_LOOP_SCENARIO_MATRIX",
    "post_test_retraining_allowed": false,
    "processing_order": [
        "raw_window",
        "feature_standardization",
        "frozen_model_inference",
        "inverse_target_scaling",
        "finite_check",
        "admissible_gain_clipping",
        "gain_rate_limiting",
        "applied_gain"
    ],
    "rate_limit": {
        "ki_max_rate_per_s": 11.1072,
        "ki_max_step_per_update": 1.1107200000000002,
        "kp_max_rate_per_s": 1.1999999999999997,
        "kp_max_step_per_update": 0.11999999999999998,
        "minimum_full_range_traverse_time_s": 1.0,
        "selected_using_closed_loop_results": false
    },
    "stage": "FROZEN_SCHEDULER_INTERFACE_AND_GAIN_POLICY"
}


SCHEDULER_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "scheduler_policy"
    ]
)


scheduler_policy = json.loads(
    SCHEDULER_PATH.read_text(
        encoding="utf-8"
    )
)


SCHEDULER_POLICY_EXACT = bool(
    scheduler_policy
    ==
    EXPECTED_SCHEDULER_POLICY
)


if not SCHEDULER_POLICY_EXACT:

    raise RuntimeError(
        "Frozen scheduler policy changed."
    )


print(
    "Fixed PI:",
    (
        scheduler_policy[
            "fixed_pi"
        ][
            "kp"
        ],
        scheduler_policy[
            "fixed_pi"
        ][
            "ki"
        ],
    )
)

print(
    "Observation rate [Hz]:",
    scheduler_policy[
        "input"
    ][
        "observation_rate_hz"
    ]
)

print(
    "Window points:",
    scheduler_policy[
        "input"
    ][
        "window_points"
    ]
)

print(
    "Gain update rate [Hz]:",
    scheduler_policy[
        "gain_update"
    ][
        "update_rate_hz"
    ]
)

print(
    "Between updates:",
    scheduler_policy[
        "gain_update"
    ][
        "between_updates"
    ]
)

print(
    "Gain envelope:",
    scheduler_policy[
        "admissible_gain_envelope"
    ]
)

print(
    "Fallback:",
    scheduler_policy[
        "fallback"
    ]
)

print(
    "Same scheduler safety policy LSTM/QLSTM:",
    all(
        scheduler_policy[
            "fairness"
        ].values()
    )
)

print(
    "\nSCHEDULER_POLICY_EXACT:",
    SCHEDULER_POLICY_EXACT
)


In [ ]:
# ====================================================
# 09.3 — PREREGISTERED SCENARIO / RUN MATRIX
# ====================================================

PRE_POLICY_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "pre_simulation_policy"
    ]
)


RUN_MATRIX_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "run_matrix"
    ]
)


SCENARIO_MATRIX_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "scenario_matrix"
    ]
)


pre_policy = json.loads(
    PRE_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


run_matrix = pd.read_csv(
    RUN_MATRIX_PATH
)


scenario_matrix = pd.read_csv(
    SCENARIO_MATRIX_PATH
)


if len(
    run_matrix
) != 12:

    raise RuntimeError(
        "Expected 12 preregistered runs."
    )


if len(
    scenario_matrix
) != 4:

    raise RuntimeError(
        "Expected 4 preregistered scenarios."
    )


print(
    "Scenario matrix:"
)


print(
    scenario_matrix.to_string(
        index=False
    )
)


print(
    "\nRun matrix:"
)


print(
    run_matrix.to_string(
        index=False
    )
)


print(
    "\nPrimary independent families:",
    pre_policy[
        "analysis_policy"
    ][
        "primary_independent_families"
    ]
)

print(
    "Primary independent family count:",
    pre_policy[
        "analysis_policy"
    ][
        "primary_independent_family_count"
    ]
)

print(
    "Window-level significance allowed:",
    pre_policy[
        "analysis_policy"
    ][
        "window_level_significance_tests_allowed"
    ]
)

print(
    "Oracle gain used for control:",
    pre_policy[
        "analysis_policy"
    ][
        "oracle_gain_used_for_control"
    ]
)


In [ ]:
# ============================================================
# 09.4 — ADAPTIVE RK4 SIMULATOR CONTRACT
# ============================================================

EXPECTED_ADAPTIVE_CONTRACT = {
    "cell153_readiness": {
        "path": "data/closed_loop/cell153_simulator_readiness.json",
        "sha256": "1f75fe58c40d8f371d194ed9bd71cf91cff835ffd332272106ac586a456d5e03"
    },
    "constant_gain_equivalence": {
        "all_pass": true,
        "gain": [
            2.16,
            5.72
        ],
        "maximum_absolute_error": 0.0,
        "scenarios": [
            "CL01",
            "CL02",
            "CL03",
            "CL04"
        ],
        "tolerance": 1e-12
    },
    "feature_contract": [
        "frequency_deviation_hz",
        "mechanical_power_kw",
        "electrical_power_kw",
        "dump_power_kw"
    ],
    "gain_application": {
        "adaptive_update_requires_time_strictly_after_disturbance": true,
        "between_updates": "ZERO_ORDER_HOLD",
        "bumpless_transfer_state_correction": false,
        "common_cell151_gain_guard": true,
        "first_adaptive_update_s": 2.6,
        "integral_reset_on_gain_change": false,
        "output_bias": "FIXED_SCENARIO_SPECIFIC_INITIAL_DUTY",
        "pi_integral_state": "CONTINUOUS_ACROSS_GAIN_UPDATES",
        "steps_per_gain_update": 40,
        "update_dt_s": 0.1,
        "update_rate_hz": 10.0,
        "updated_gain_applies_from_update_instant_forward": true
    },
    "next_cell": "CELL_155_FROZEN_CLOSED_LOOP_COMPARATIVE_EXPERIMENT",
    "observation": {
        "collected_from_simulation_start": true,
        "dt_s": 0.05,
        "rate_hz": 20.0,
        "steps_per_observation": 20,
        "window_points": 11,
        "window_span_s": 0.5
    },
    "pre_simulation_checkpoint": "ec2b0eb88da803dbbfd0ad7c58c107e857b2be47",
    "rk4": {
        "controller_gain_constant_within_each_rk4_step": true,
        "dt_s": 0.0025,
        "k4_event_semantics": "LEFT_LIMIT_NEXTAFTER",
        "method": "FIXED_STEP_RK4",
        "uses_project_closed_loop_rhs": true
    },
    "scientific_policy": {
        "controller_comparison_performed": false,
        "lstm_closed_loop_inference_performed": false,
        "model_retraining_allowed": false,
        "performance_ranking_performed": false,
        "qlstm_closed_loop_inference_performed": false,
        "rmse_comparison_performed": false,
        "scheduler_policy_change_after_comparison_allowed": false,
        "simulator_algorithm_change_after_comparison_allowed": false
    },
    "scripted_structural_smoke": {
        "actuator_domain_ok": true,
        "all_states_finite": true,
        "cadence_ok": true,
        "first_update_ok": true,
        "gain_bounds_ok": true,
        "gain_step_limit_ok": true,
        "integral_reset_applied": false,
        "model_inference_used": false,
        "no_pre_event_gain_change": true,
        "requested_gain": [
            3.36,
            14.144
        ],
        "scenario": "CL01",
        "smoke_pass": true,
        "zero_order_hold_ok": true
    },
    "stage": "ADAPTIVE_RK4_SIMULATOR_INTEGRATION_FREEZE"
}


ADAPTIVE_CONTRACT_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "adaptive_contract"
    ]
)


adaptive_contract = json.loads(
    ADAPTIVE_CONTRACT_PATH.read_text(
        encoding="utf-8"
    )
)


ADAPTIVE_CONTRACT_EXACT = bool(
    adaptive_contract
    ==
    EXPECTED_ADAPTIVE_CONTRACT
)


if not ADAPTIVE_CONTRACT_EXACT:

    raise RuntimeError(
        "Adaptive RK4 contract changed."
    )


print(
    "Solver:",
    adaptive_contract[
        "rk4"
    ][
        "method"
    ]
)

print(
    "RK4 dt [s]:",
    adaptive_contract[
        "rk4"
    ][
        "dt_s"
    ]
)

print(
    "Event semantics:",
    adaptive_contract[
        "rk4"
    ][
        "k4_event_semantics"
    ]
)

print(
    "Constant-gain equivalence PASS:",
    adaptive_contract[
        "constant_gain_equivalence"
    ][
        "all_pass"
    ]
)

print(
    "Maximum equivalence error:",
    adaptive_contract[
        "constant_gain_equivalence"
    ][
        "maximum_absolute_error"
    ]
)

print(
    "First adaptive gain update [s]:",
    adaptive_contract[
        "gain_application"
    ][
        "first_adaptive_update_s"
    ]
)

print(
    "PI integral continuous:",
    adaptive_contract[
        "gain_application"
    ][
        "pi_integral_state"
    ]
)

print(
    "Integral reset on gain change:",
    adaptive_contract[
        "gain_application"
    ][
        "integral_reset_on_gain_change"
    ]
)

print(
    "Bumpless correction:",
    adaptive_contract[
        "gain_application"
    ][
        "bumpless_transfer_state_correction"
    ]
)

print(
    "\nADAPTIVE_CONTRACT_EXACT:",
    ADAPTIVE_CONTRACT_EXACT
)


In [ ]:
# ====================================================
# 09.5 — LOAD FROZEN 12-RUN RESULTS
# ====================================================

METRICS_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "metrics"
    ]
)


EXPERIMENT_METADATA_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "experiment_metadata"
    ]
)


PRIMARY_SUMMARY_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "primary_summary"
    ]
)


PRIMARY_WINNERS_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "primary_winners"
    ]
)


metrics = pd.read_csv(
    METRICS_PATH
)


experiment_metadata = json.loads(
    EXPERIMENT_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)


primary_summary = pd.read_csv(
    PRIMARY_SUMMARY_PATH
)


primary_winners = pd.read_csv(
    PRIMARY_WINNERS_PATH
)


if len(
    metrics
) != 12:

    raise RuntimeError(
        "Expected exactly 12 frozen runs."
    )


structural_columns = [
    "rows_ok",
    "all_numeric_finite",
    "gain_bounds_ok",
    "actuator_domain_ok",
    "pre_adaptive_fixed_gain_ok",
    "update_contract_ok",
    "run_structure_ok",
]


for column in structural_columns:

    if not bool(
        metrics[
            column
        ].all()
    ):

        raise RuntimeError(
            f"Closed-loop structural failure: {column}"
        )


print(
    "Frozen closed-loop metrics:"
)


display_columns = [
    "run_id",
    "scenario_id",
    "family_key",
    "evaluation_role",
    "controller",
    "frequency_deviation_rmse_hz",
    "peak_abs_frequency_deviation_hz",
    "settling_time_s",
    "max_abs_rocof_hz_per_s",
    "dump_duty_total_variation",
    "simulation_runtime_s",
]


print(
    metrics[
        display_columns
    ].to_string(
        index=False
    )
)


print(
    "\nAll runs structurally valid:",
    bool(
        metrics[
            "run_structure_ok"
        ].all()
    )
)


print(
    "Experiment complete:",
    experiment_metadata[
        "experiment_complete"
    ]
)


CLOSED_LOOP_RESULTS_STRUCTURALLY_VALID = True


In [ ]:
# ====================================================
# 09.6 — RAW TRAJECTORY + GAIN-UPDATE AUDIT
# ====================================================

TRAJECTORIES_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "trajectories"
    ]
)


ADAPTIVE_UPDATES_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "adaptive_updates"
    ]
)


trajectory_rows = 0


for chunk in pd.read_csv(
    TRAJECTORIES_PATH,
    compression="gzip",
    chunksize=100000,
):

    trajectory_rows += len(
        chunk
    )


adaptive_updates = pd.read_csv(
    ADAPTIVE_UPDATES_PATH,
    compression="gzip",
)


expected_trajectory_rows = (
    12
    *
    4001
)


if (
    trajectory_rows
    !=
    expected_trajectory_rows
):

    raise RuntimeError(
        "Frozen trajectory row count changed."
    )


if (
    len(
        adaptive_updates
    )
    !=
    experiment_metadata[
        "results"
    ][
        "adaptive_update_records"
    ]
):

    raise RuntimeError(
        "Adaptive update record count changed."
    )


print(
    "Trajectory rows:",
    trajectory_rows
)

print(
    "Expected rows:",
    expected_trajectory_rows
)

print(
    "Adaptive gain-update records:",
    len(
        adaptive_updates
    )
)

print(
    "Experiment runtime [s]:",
    experiment_metadata[
        "results"
    ][
        "total_experiment_runtime_s"
    ]
)


In [ ]:
# ====================================================
# 09.7 — PRIMARY DESCRIPTIVE COMPARISON
# ====================================================

print(
    primary_summary.to_string(
        index=False
    )
)


print(
    "\nPer-family lowest-RMSE controller:"
)


print(
    primary_winners.to_string(
        index=False
    )
)


ranking = (
    primary_summary
    .sort_values(
        "rank_by_mean_primary_rmse"
    )[
        "controller"
    ]
    .tolist()
)


if ranking != [
    "FIXED_PI",
    "LSTM_PI",
    "QLSTM_PI",
]:

    raise RuntimeError(
        "Frozen primary descriptive ranking changed."
    )


winner_pairs = set(
    zip(
        primary_winners[
            "scenario_id"
        ],
        primary_winners[
            "lowest_rmse_controller"
        ],
    )
)


if winner_pairs != {
    ("CL01", "FIXED_PI"),
    ("CL02", "LSTM_PI"),
}:

    raise RuntimeError(
        "Frozen primary per-family winners changed."
    )


FIXED_PRIMARY_RMSE = float(
    primary_summary.loc[
        primary_summary[
            "controller"
        ]
        ==
        "FIXED_PI",
        "mean_primary_rmse_hz",
    ].iloc[0]
)


LSTM_PRIMARY_RMSE = float(
    primary_summary.loc[
        primary_summary[
            "controller"
        ]
        ==
        "LSTM_PI",
        "mean_primary_rmse_hz",
    ].iloc[0]
)


QLSTM_PRIMARY_RMSE = float(
    primary_summary.loc[
        primary_summary[
            "controller"
        ]
        ==
        "QLSTM_PI",
        "mean_primary_rmse_hz",
    ].iloc[0]
)


print(
    "\nFixed PI mean primary RMSE [Hz]:",
    FIXED_PRIMARY_RMSE
)

print(
    "LSTM-PI mean primary RMSE [Hz]:",
    LSTM_PRIMARY_RMSE
)

print(
    "QLSTM-PI mean primary RMSE [Hz]:",
    QLSTM_PRIMARY_RMSE
)


print(
    "\nIMPORTANT:"
)

print(
    "This primary aggregate is descriptive only."
)

print(
    "Independent primary family count = 2."
)

print(
    "No statistical significance claim is made."
)


STATISTICAL_SIGNIFICANCE_CLAIM_ALLOWED = False
QLSTM_CLOSED_LOOP_SUPERIORITY_SUPPORTED = False


In [ ]:
# ====================================================
# 09.8 — CLOSED-LOOP SOURCE ARCHIVE AUDIT
# ====================================================

snapshot = json.loads(
    RUNTIME_SOURCE_SNAPSHOT_PATH.read_text(
        encoding="utf-8"
    )
)


expected_labels = [
    "151",
    "152",
    "153",
    "154",
    "155",
]


SOURCE_ARCHIVE_COMPLETE = bool(
    snapshot[
        "labels"
    ]
    ==
    expected_labels

    and

    all(
        label
        in
        snapshot[
            "cells"
        ]

        for label
        in
        expected_labels
    )
)


print(
    "Archived runtime cells:"
)


for label in expected_labels:

    record = (
        snapshot[
            "cells"
        ][
            label
        ]
    )


    print(
        f"CELL {label} | "
        f"{record['source_chars']:7d} chars | "
        f"{record['source_sha256']}"
    )


print(
    "\nSOURCE_ARCHIVE_COMPLETE:",
    SOURCE_ARCHIVE_COMPLETE
)


if not SOURCE_ARCHIVE_COMPLETE:

    raise RuntimeError(
        "Closed-loop runtime source archive incomplete."
    )


## 3. Kebijakan Reproduksi Eksperimen

Frozen 12-run experiment merupakan hasil primer yang sudah dieksekusi
dan dikunci.

Notebook migrasi **tidak menjalankan ulang eksperimen secara default**.

Source asli tersedia dalam archive Cell 151–155 sehingga reproduksi
penuh tetap dapat dilakukan sebagai pekerjaan terpisah, tetapi tidak
digunakan untuk mengganti hasil primer setelah hasil telah diketahui.


In [ ]:
# ====================================================
# 09.9 — EXPLICIT CLOSED-LOOP RERUN GUARD
# ====================================================

RERUN_CLOSED_LOOP_EXPERIMENT = False

CLOSED_LOOP_EXPERIMENT_RERUN = False

MODEL_INFERENCE_PERFORMED = False


if not RERUN_CLOSED_LOOP_EXPERIMENT:

    print(
        "RERUN_CLOSED_LOOP_EXPERIMENT = False"
    )

    print(
        "Frozen Cell-155 12-run experiment "
        "remains authoritative."
    )

    print(
        "No simulator or model inference was executed."
    )


else:

    raise RuntimeError(
        "Automatic rerun is intentionally disabled "
        "in the migrated verification notebook. "
        "The exact Cell-151–155 source is archived. "
        "A controlled reproduction must be run as a "
        "separate reproduction workflow without "
        "changing models, scenarios, metrics, scheduler "
        "policy, or simulator algorithm."
    )


## 4. Handoff ke `10_final_analysis.ipynb`

Notebook 09 berakhir pada **frozen raw closed-loop results**.

Notebook berikutnya melakukan:

- audit hasil;
- interpretasi hubungan model-error dan control-performance;
- computational-burden audit;
- scientific-claim freeze;
- tabel dan gambar tesis;
- penyusunan Bab IV.

`10_final_analysis.ipynb` tidak boleh mengubah kembali simulator,
scheduler, model, ataupun scenario matrix.


In [ ]:
# ====================================================
# 09.10 — CLOSED-LOOP STAGE SUMMARY
# ====================================================

NOTEBOOK_09_CLOSED_LOOP_STAGE_READY = all(
    [
        FROZEN_CLOSED_LOOP_ARTIFACTS_VALID,
        SCHEDULER_POLICY_EXACT,
        ADAPTIVE_CONTRACT_EXACT,
        CLOSED_LOOP_RESULTS_STRUCTURALLY_VALID,
        SOURCE_ARCHIVE_COMPLETE,
        not STATISTICAL_SIGNIFICANCE_CLAIM_ALLOWED,
        not QLSTM_CLOSED_LOOP_SUPERIORITY_SUPPORTED,
        not CLOSED_LOOP_EXPERIMENT_RERUN,
        not MODEL_INFERENCE_PERFORMED,
    ]
)


print("=" * 72)
print("09_closed_loop.ipynb — SUMMARY")
print("=" * 72)


print(
    "Controllers                      : 3"
)

print(
    "Scenarios                        : 4"
)

print(
    "Frozen deterministic runs        : 12"
)

print(
    "Primary independent families     : 2"
)

print(
    "RK4 dt [s]                       :",
    adaptive_contract[
        "rk4"
    ][
        "dt_s"
    ]
)

print(
    "Observation rate [Hz]             :",
    scheduler_policy[
        "input"
    ][
        "observation_rate_hz"
    ]
)

print(
    "Gain update rate [Hz]             :",
    scheduler_policy[
        "gain_update"
    ][
        "update_rate_hz"
    ]
)

print(
    "First adaptive update [s]         :",
    adaptive_contract[
        "gain_application"
    ][
        "first_adaptive_update_s"
    ]
)

print(
    "Constant-gain equivalence error   :",
    adaptive_contract[
        "constant_gain_equivalence"
    ][
        "maximum_absolute_error"
    ]
)

print(
    "Trajectory rows                  :",
    trajectory_rows
)

print(
    "Adaptive update records          :",
    len(
        adaptive_updates
    )
)

print(
    "Fixed PI mean primary RMSE [Hz]  :",
    FIXED_PRIMARY_RMSE
)

print(
    "LSTM-PI mean primary RMSE [Hz]   :",
    LSTM_PRIMARY_RMSE
)

print(
    "QLSTM-PI mean primary RMSE [Hz]  :",
    QLSTM_PRIMARY_RMSE
)

print(
    "Descriptive mean winner           : FIXED_PI"
)

print(
    "CL01 winner                       : FIXED_PI"
)

print(
    "CL02 winner                       : LSTM_PI"
)

print(
    "Statistical significance claimed  : False"
)

print(
    "QLSTM closed-loop superiority     : False"
)

print(
    "Post-result tuning                : False"
)

print(
    "Experiment rerun requested        :",
    RERUN_CLOSED_LOOP_EXPERIMENT
)

print(
    "Experiment rerun performed        :",
    CLOSED_LOOP_EXPERIMENT_RERUN
)

print(
    "Model inference in verify mode    :",
    MODEL_INFERENCE_PERFORMED
)

print(
    "Runtime source archive complete   :",
    SOURCE_ARCHIVE_COMPLETE
)

print(
    "NOTEBOOK 09 CLOSED-LOOP READY     :",
    NOTEBOOK_09_CLOSED_LOOP_STAGE_READY
)


if NOTEBOOK_09_CLOSED_LOOP_STAGE_READY:

    print(
        "\nNEXT NOTEBOOK:"
    )

    print(
        "10_final_analysis.ipynb"
    )
